In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
link="/Volumes/aws_catalog/default/aws_databricks/project/"

df=spark.read.format("csv").option("inferschema","true")\
    .option("header","true").load(f"{link}/customers/*.csv")\
    .withColumn("ingestion_time", current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")
    


display(df)

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.bronze.customers")

In [0]:
df_silver=spark.read.table("fmcg.bronze.customers")
display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
df_silver=df_silver.dropDuplicates(["customer_id"])
df_silver.count()

In [0]:
df_silver=df_silver.withColumn("customer_id", col("customer_id").cast("string"))
df_silver.printSchema()

In [0]:
df_silver=df_silver.withColumn("customer_name",trim(col("customer_name")))
df_silver=df_silver.withColumn("customer_name", initcap(col("customer_name")))
display(df_silver)

In [0]:
display(df.select("city").distinct())

In [0]:
mapping={
    "Hyderbad":"Hyderabad",
    "Hyderabadd":"Hyderabad",
    "NewDelhee":"New Delhi",
    "NewDheli":"New Delhi",
    "NewDehli":"New Delhi",
    "NewDelhi":"New Delhi",
    "Bengaluruu":"Bengaluru",
    "Bengalore":"Bengaluru"
    }

In [0]:
df_silver=df_silver.replace(mapping, subset=["city"])
df_silver.select("city").distinct().show()

In [0]:
display(df_silver)

In [0]:
display(df_silver.filter(col("city").isNull()))

In [0]:
display(df_silver.filter(col("customer_name")=="Sprintx Nutrition"))

In [0]:
display(df_silver.filter(col("customer_name")=="Zenathlete Foods"))

In [0]:
display(df_silver.filter(col("customer_name")=="Primefuel Nutrition"))

In [0]:
display(df_silver.filter(col("customer_name")=="Recovery Lane"))

In [0]:
city_mapping={
    # Sprintx Nutrition
    789403:"New Delhi",
    # Zenathlete Foods
    789420:"Bengaluru",
    # Primefuel Nutrition
    789521:"Hyderabad",
    # Recovery Lane
    789603:"Hyderabad"
}

In [0]:
cities=spark.createDataFrame(city_mapping.items(), ["customer_id", "fix_city"])
display(cities)


In [0]:
df_silver=df_silver.join(cities, on="customer_id", how="left")
df_silver=df_silver.withColumn("city", coalesce(col("city"),col("fix_city")))
df_silver=df_silver.drop("fix_city")
display(df_silver)



In [0]:
df_silver=df_silver.withColumn("customer", concat_ws("-", col("customer_name"), coalesce(col("city"),lit("unknown"))))\
    .withColumn('market', lit("india"))\
    .withColumn("platform",lit("Sports Bar"))\
    .withColumn("channel",lit("Aquisition"))
display(df_silver.limit(5))

In [0]:
df_silver.write.format("delta").option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.silver.customers")

In [0]:
df_gold=spark.read.table("fmcg.silver.customers")
df_gold=df_gold.select("customer_id","customer_name","city","customer","market","platform","channel")
display(df_gold)

In [0]:
df_gold.write.format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable("fmcg.gold.sb_dim_customers")

In [0]:
from delta.tables import DeltaTable

In [0]:
parent_table=DeltaTable.forName(spark,"fmcg.gold.dim_customers")
child_table=spark.read.table("fmcg.gold.sb_dim_customers").select(col("customer_id").alias("customer_code"),"customer_name","city","customer","market","platform","channel")

(parent_table.alias("target")
 .merge(
     child_table.alias("source"),
     "target.customer_code = source.customer_code"
 )
 .whenMatchedUpdateAll()     # Updates existing matching customers
 .whenNotMatchedInsertAll()  # Inserts brand new customers
 .execute())



In [0]:
%sql
select * from fmcg.gold.dim_customers 